# From Logistic Regression to a Neural Network - Exercise

**Dataset:** 30,000 emails represented by 16 numerical features, a label and fixed training/validation/test membership. **0 = ham, 1 = spam**. The business decision is whether an email should be flagged for human review, not automatically deleted.

In this notebook, we will:
1. Prepare identical, training-scaled features for both models.
2. Reproduce logistic regression's sigmoid output and train a small MLP.
3. Select a hidden-layer size and decision thresholds using validation data.
4. Interpret historical test evidence, business costs and limitations.

Read the [concept notebook](logistic_regression_to_neural_network_concept.ipynb) first. Run cells from top to bottom, replacing the **four `None` values marked with `# TODO`** using the instructions above their cells. Those cells are intentionally incomplete and cannot be used until the gaps are filled. Data preparation, candidate-training loops, evaluation and plotting are provided. Write one or two sentences for each **Your interpretation** prompt, then check your work with the [solution](logistic_regression_to_neural_network_solution.ipynb).

This accompanies **SW07 Lecture 1** and assumes logistic regression, train/validation/test splits and basic Python. One worked step fits the lecture's 10-minute demonstration slot; the complete exercise belongs in a separate practical session. The core uses only the repository's base environment and local data. Optional XGBoost, model persistence, embeddings and a deeper network are not needed here.

## Libraries and settings

Import the libraries, set the random seed and locate the local email data. The same settings and prepared features are used throughout the main comparison.

In [ ]:
from pathlib import Path
import importlib.metadata
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 7
LABEL_MEANINGS = {0: 'ham', 1: 'spam'}
np.random.seed(SEED)
search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((root for root in search_roots
                  if (root / '00_Data' / 'email_spam.csv').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside python_machine_learning_basics; 00_Data/email_spam.csv must be present.')
DATA_DIR = REPO_ROOT / '00_Data'
NOTEBOOK_DIR = REPO_ROOT / '08_Neural_Networks' / 'HS2026'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from enron_features import EmailFeatures, FEATURE_NAMES

versions = {package: importlib.metadata.version(package)
            for package in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']}
print('Data directory:', DATA_DIR)
print('Documented labels:', LABEL_MEANINGS)
display(pd.Series(versions, name='Installed version'))

## 1. Import the prepared email data

The supplied code reads the local CSV explicitly and keeps its stored **18,000 training, 6,000 validation and 6,000 test rows**. One observation is one email; labels are 0 = ham and 1 = spam. No raw subjects or bodies are included.

Preparation removed empty and normalized exact-duplicate messages and quarantined conflicting labels before sampling with seed 7. Related campaigns and near-duplicates may remain across splits. Do not resplit this reference data.

**Question:** Which four columns must not become predictors? Why do we retain the stored split membership?

**Your interpretation:** Replace this sentence with your answer.

In [ ]:
email_data_path = DATA_DIR / 'email_spam.csv'
prepared_emails = pd.read_csv(email_data_path)

train = prepared_emails.loc[prepared_emails['split'].eq('train')].copy()
validation = prepared_emails.loc[prepared_emails['split'].eq('validation')].copy()
test = prepared_emails.loc[prepared_emails['split'].eq('test')].copy()

print('Loaded:', email_data_path.resolve())
print(f'Ready: {len(train):,} training, {len(validation):,} validation and {len(test):,} test emails.')

In [ ]:
display(pd.DataFrame({
    'Split': ['Training', 'Validation', 'Test'],
    'Emails': [len(train), len(validation), len(test)],
    'Purpose': ['Fit models', 'Choose settings and thresholds', 'Evaluate fixed choices'],
}))

## 2. Select and scale shared inputs

Both models receive the same 16 count and ratio features: message size, uppercase/digit ratios, punctuation, symbols, links, subject length and six promotional-word counts. The supplied helper selects those features in a fixed order; it can also compute them from new subject/body inputs. These inputs describe patterns, not meaning.

**Coding task:** in the preprocessing cell, obtain `X_train` by fitting and transforming `train_messages` with `preprocessing`, passing `y_train`. Validation and test transformations are supplied and must reuse that fitted pipeline. Preparation and histogram loops are provided.

**Question:** Why must the scaler see training data only? Why must we not fit a separate scaler on validation or test data?

**Your interpretation:** Replace this sentence with your answer.

In [ ]:
print('Select shared email features')
content_columns = list(FEATURE_NAMES)
train_messages = train[content_columns].copy()
validation_messages = validation[content_columns].copy()
test_messages = test[content_columns].copy()
y_train = train['label'].to_numpy()
y_validation = validation['label'].to_numpy()
y_test = test['label'].to_numpy()
feature_transformer = EmailFeatures()
raw_train_features = feature_transformer.fit_transform(train_messages)
display(raw_train_features.head())
display(raw_train_features.agg(['min', 'median', 'max']).T)
print('Inputs contain only:', train_messages.columns.tolist())

### Scale the input features

Fit the feature extractor and scaler on training emails only, then transform validation and test emails with the same fitted pipeline. The histograms describe selected training features; they do not guide changes using test data.

In [ ]:
print('Fit shared preprocessing on training data only')
preprocessing = Pipeline([('features', EmailFeatures()), ('scale', StandardScaler())])
# TODO: Fit preprocessing on the training messages and labels.
X_train = None
X_validation = preprocessing.transform(validation_messages)
X_test = preprocessing.transform(test_messages)
feature_names = preprocessing.named_steps['features'].get_feature_names_out()
print('Train / validation / test shapes:', X_train.shape, X_validation.shape, X_test.shape)
figure, axes = plt.subplots(1, 3, figsize=(12, 3.3), constrained_layout=True)
for axis, feature in zip(axes, ['word_count', 'uppercase_letter_ratio', 'exclamation_count']):
    for label, name in LABEL_MEANINGS.items():
        values = raw_train_features.loc[y_train == label, feature]
        if feature != 'uppercase_letter_ratio':
            values = np.log1p(values)
        axis.hist(values, bins=35, density=True, alpha=0.45, label=name)
    axis.set_title(feature)
    axis.set_xlabel('Raw ratio' if feature == 'uppercase_letter_ratio' else 'log(1 + count), plot only')
    axis.set_ylabel('Density')
axes[0].legend(fontsize=8)
plt.show()

## 3. Fit logistic regression and reproduce its output

**Coding task:** create `logistic` as a `LogisticRegression` estimator with `C=1.0`, `max_iter=1500` and `random_state=SEED`. Fitting, warning capture, a training-prior baseline, validation metrics and plots are supplied.

Logistic regression uses $z=\mathbf{w}^{T}\mathbf{x}+b$ followed by $p=1/(1+\exp(-z))$. Here $\mathbf{x}$ contains the scaled inputs, $\mathbf{w}$ the learned weights and $b$ the bias. The manual calculation below reproduces prediction, not training.

**Question:** How many trainable parameters are there? Does the manual sigmoid agree with `predict_proba`? Can a positive feature weight be interpreted as a causal effect?

**Your interpretation:** Replace this sentence with your answer.

In [ ]:
print('Logistic regression as a single output neuron')
# TODO: Construct logistic regression with the requested settings.
logistic = None
with warnings.catch_warnings(record=True) as logistic_warnings:
    warnings.simplefilter('always', ConvergenceWarning)
    logistic.fit(X_train, y_train)
for warning in logistic_warnings:
    print(type(warning.message).__name__, str(warning.message))
weighted_sum = X_validation[:5] @ logistic.coef_[0] + logistic.intercept_[0]
manual_probability = np.exp(-np.logaddexp(0, -weighted_sum))
print('Parameters:', logistic.coef_.size + logistic.intercept_.size)
display(pd.DataFrame({'weighted_sum': weighted_sum, 'manual_sigmoid': manual_probability,
                      'predict_proba': logistic.predict_proba(X_validation[:5])[:, 1]}))

In [ ]:
print('Validation evaluation and feature weights')


def metrics_at_threshold(targets, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {'precision': precision_score(targets, predictions, zero_division=0),
            'recall': recall_score(targets, predictions, zero_division=0),
            'f1': f1_score(targets, predictions, zero_division=0),
            'average_precision': average_precision_score(targets, probabilities),
            'flagged_fraction': float(predictions.mean())}


baseline = DummyClassifier(strategy='prior', random_state=SEED).fit(X_train, y_train)
validation_probabilities = {
    'Prior baseline': baseline.predict_proba(X_validation)[:, 1],
    'Logistic regression': logistic.predict_proba(X_validation)[:, 1],
}
display(pd.DataFrame({name: metrics_at_threshold(y_validation, probabilities)
                      for name, probabilities in validation_probabilities.items()}).T)
figure, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ConfusionMatrixDisplay.from_predictions(y_validation, validation_probabilities['Logistic regression'] >= 0.5,
                                        display_labels=['ham', 'spam'], ax=axes[0], colorbar=False)
axes[0].set_title('Validation: logistic regression at 0.5')
coefficients = pd.Series(logistic.coef_[0], index=feature_names).sort_values()
coefficient_subset = pd.concat([coefficients.head(5), coefficients.tail(5)]).drop_duplicates().sort_values()
coefficient_subset.plot.barh(ax=axes[1], color=['#C34800' if value > 0 else '#00776E' for value in coefficient_subset])
axes[1].axvline(0, color='black', linewidth=0.7)
axes[1].set_title('Standardized logistic coefficients')
axes[1].set_xlabel('Conditional change in log-odds per training standard deviation')
plt.show()
print('Coefficients describe model associations, not causal effects. Correlated counts make attribution unstable.')

## 4. Train and select a hidden layer

The supplied loop trains **16 inputs -> 8 or 16 ReLU units -> one sigmoid output**, with the original seed, Adam optimizer, regularization, batch size and epoch budget. Backpropagation supplies gradients for training; its calculation is covered in SW07 Lecture 2.

**Coding task:** set `best_size` to the integer index of the largest `validation_ap` in `network_comparison`. Candidate training, plotting and a manual forward-pass check are supplied. Select on validation AP, not test scores or the lowest training loss.

**Question:** What changed compared with logistic regression? How many parameters does the 8-unit network have? Which candidate is selected, and what do the warnings allow us to conclude?

**Your interpretation:** Replace this sentence with your answer.

In [ ]:
network_candidates = {}
network_records = []
for hidden_size in [8, 16]:
    candidate = MLPClassifier(hidden_layer_sizes=(hidden_size,), activation='relu', solver='adam',
                              alpha=0.01, batch_size=128, learning_rate_init=0.001,
                              max_iter=200, n_iter_no_change=15, early_stopping=False,
                              random_state=SEED)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        candidate.fit(X_train, y_train)
    messages = [str(warning.message) for warning in caught]
    print(f'Hidden size {hidden_size}: {candidate.n_iter_} epochs; warnings: {messages or "none"}')
    probabilities = candidate.predict_proba(X_validation)[:, 1]
    network_candidates[hidden_size] = candidate
    network_records.append({'hidden_units': hidden_size, 'epochs': candidate.n_iter_,
                            'validation_ap': average_precision_score(y_validation, probabilities),
                            'warnings': len(messages)})
network_comparison = pd.DataFrame(network_records).set_index('hidden_units')
display(network_comparison)
# TODO: Select the integer hidden size with the largest validation AP.
best_size = None
network = network_candidates[best_size]
print('Selected hidden size:', best_size)
print('Weight shapes:', [weights.shape for weights in network.coefs_])
print('Bias shapes:', [bias.shape for bias in network.intercepts_])
print('Output activation:', network.out_activation_)
print('MLP parameters:', sum(weights.size for weights in network.coefs_) + sum(bias.size for bias in network.intercepts_))
hidden = np.maximum(0, X_validation[:5] @ network.coefs_[0] + network.intercepts_[0])
output_score = hidden @ network.coefs_[1] + network.intercepts_[1]
manual_network_probability = np.exp(-np.logaddexp(0, -output_score)).ravel()
np.testing.assert_allclose(manual_network_probability, network.predict_proba(X_validation[:5])[:, 1])
validation_probabilities['Neural network'] = network.predict_proba(X_validation)[:, 1]
display(pd.DataFrame({name: metrics_at_threshold(y_validation, probabilities)
                      for name, probabilities in validation_probabilities.items()}).T.round(4))
figure, axis = plt.subplots(figsize=(7, 3.4), constrained_layout=True)
for size, candidate in network_candidates.items():
    axis.plot(np.arange(1, len(candidate.loss_curve_) + 1), candidate.loss_curve_, label=f'{size} hidden units')
axis.set(xlabel='Epoch', ylabel='Training objective', title='Optimization curves, not validation performance')
axis.legend()
plt.show()

## 5. Select thresholds using validation costs

Assume a false positive costs **5 units** and a false negative costs **1 unit**. Flag an email when its probability reaches the threshold. The supplied code searches thresholds 0.05 to 0.95 in steps of 0.01, using validation data only. The no-skill baseline keeps threshold 0.5.

**Coding task:** calculate `cost` from `FALSE_POSITIVE_COST`, `FALSE_NEGATIVE_COST`, `false_positive` and `false_negative`. The search and comparison table are provided.

**Question:** Why can the highest-AP model still be a poor business policy? Why must we not choose a threshold from the test results? What if the selected value is the edge of the search grid?

**Your interpretation:** Replace this sentence with your answer.

In [ ]:
FALSE_POSITIVE_COST = 5
FALSE_NEGATIVE_COST = 1
models = {'Prior baseline': baseline, 'Logistic regression': logistic,
          'Neural network': network}
validation_probabilities = {name: model.predict_proba(X_validation)[:, 1]
                            for name, model in models.items()}
thresholds = {'Prior baseline': 0.5}
threshold_records = []
for name in ['Logistic regression', 'Neural network']:
    probabilities = validation_probabilities[name]
    candidates = []
    for threshold in np.linspace(0.05, 0.95, 91):
        true_negative, false_positive, false_negative, true_positive = confusion_matrix(
            y_validation, probabilities >= threshold, labels=[0, 1]).ravel()
        # TODO: Calculate the weighted validation error cost.
        cost = None
        candidates.append((cost, float(threshold)))
    best_cost, best_threshold = min(candidates)
    thresholds[name] = best_threshold
for name, probabilities in validation_probabilities.items():
    selected_threshold = thresholds[name]
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        y_validation, probabilities >= selected_threshold, labels=[0, 1]).ravel()
    threshold_records.append({'model': name, 'threshold': selected_threshold,
                              'cost_per_1000': 1000 * (FALSE_POSITIVE_COST * false_positive + FALSE_NEGATIVE_COST * false_negative) / len(y_validation),
                              **metrics_at_threshold(y_validation, probabilities, selected_threshold)})
display(pd.DataFrame(threshold_records).set_index('model').round(4))
print('Frozen architecture:', network.hidden_layer_sizes, '| thresholds:', thresholds)

## 6. Evaluate fixed choices and recommend a policy

The supplied tables and plots compare the prior baseline, logistic regression and selected MLP using their frozen validation-selected thresholds. Error inspection shows only numerical features and source-row numbers, not raw email text. The first three errors in fixed test order are illustrations, not representative error samples.

**Question:** Which policy has the lowest assumed cost per 1,000 emails? How do precision and recall qualify that recommendation? Does a cheap policy that flags nothing provide useful spam screening? Should we change a model after inspecting these test errors?

**Your interpretation:** Replace this sentence with your answer.

In [ ]:
test_probabilities = {name: model.predict_proba(X_test)[:, 1] for name, model in models.items()}
test_records = []
for name, probabilities in test_probabilities.items():
    threshold = thresholds[name]
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        y_test, probabilities >= threshold, labels=[0, 1]).ravel()
    test_records.append({'model': name, 'threshold': threshold,
                         **metrics_at_threshold(y_test, probabilities, threshold),
                         'false_positives': false_positive, 'false_negatives': false_negative,
                         'cost_per_1000': 1000 * (FALSE_POSITIVE_COST * false_positive + FALSE_NEGATIVE_COST * false_negative) / len(y_test)})
test_results = pd.DataFrame(test_records).set_index('model')
display(test_results.round(4))
figure, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for name, probabilities in test_probabilities.items():
    precision, recall, curve_thresholds = precision_recall_curve(y_test, probabilities)
    axes[0].step(recall, precision, where='post',
                 label=f'{name}: AP={average_precision_score(y_test, probabilities):.3f}')
axes[0].axhline(y_test.mean(), color='gray', linestyle=':', label='Test prevalence')
axes[0].set(xlabel='Recall', ylabel='Precision', title='Held-out precision-recall curves', xlim=(0, 1), ylim=(0, 1.02))
axes[0].legend(fontsize=8)
for axis, name in zip(axes[1:], ['Logistic regression', 'Neural network']):
    ConfusionMatrixDisplay.from_predictions(y_test, test_probabilities[name] >= thresholds[name],
                                            display_labels=['ham', 'spam'], ax=axis, colorbar=False)
    axis.set_title(f'{name}, threshold={thresholds[name]:.2f}')
plt.show()
print('Historical test results are reused from earlier work, not fresh independent confirmation.')
print('Do not retune on these results. They do not establish future-email performance or causal effects.')

In [ ]:
print('Inspect limited false positives and false negatives without exposing email text')
network_predictions = test_probabilities['Neural network'] >= thresholds['Neural network']
error_rows = test[['label', 'source_row']].copy()
error_rows['spam_probability'] = test_probabilities['Neural network']
error_rows['predicted_label'] = network_predictions.astype(int)
error_features = preprocessing.named_steps['features'].transform(test_messages)
for error_name, mask in [('False positives', (y_test == 0) & network_predictions),
                         ('False negatives', (y_test == 1) & ~network_predictions)]:
    positions = np.flatnonzero(mask)[:3]
    print(error_name, '- first three in fixed test order, not representative samples')
    display(error_rows.iloc[positions].join(error_features[['word_count', 'uppercase_letter_ratio',
                                                           'url_count', 'term_free_count']]))
print('Raw subject/body content is withheld rather than incompletely redacted. Source rows enable authorized local review.')

## Summary and further exercises

- Logistic regression uses one sigmoid output; an MLP adds hidden ReLU units while keeping the same 16 input features.
- Fit preprocessing and models only on training data. Use validation evidence to select the hidden size and cost-based thresholds.
- Read AP, precision, recall and assumed cost together. An all-ham baseline can look inexpensive without finding any spam.
- Learned coefficients and neural-network weights describe associations, not causal effects. Correlated features complicate interpretation.
- Simple counts miss meaning; subjects can be absent and parsing imperfect. Near-duplicates and related campaigns can cross random splits. Historical test reuse is not fresh confirmation of future performance.
- Protect data and model artifacts. Features and hashed identifiers are not a guarantee of anonymity. Operational use needs probability checks, review-capacity analysis, time-ordered evaluation, human oversight and monitoring.

### Optional practical exercises

Possible extensions beyond the core exercise:
1. Compare XGBoost on the same features; it is a tree model, not a neural network. Its native library and Python package are additional dependencies.
2. Save preprocessing, classifier and threshold together and check synthetic messages. Load only trusted joblib artifacts. Synthetic probes reveal possible failures, not benchmark performance.
3. Compare frozen text embeddings or an extra hidden layer. Embeddings require an additional environment, authorized private raw text and verified reference split identities. The feature-only CSV cannot reconstruct email text.
4. Compare a predeclared count transformation or TF-IDF baseline. TF-IDF requires private raw text. Change one factor at a time and plan fresh evaluation before using old test findings to guide improvements.

**Transfer question:** Suppose spam campaigns change next month. What evidence would you require before automatically deleting high-scoring messages?

**Your interpretation:** Replace this sentence with your answer.

## References

**Required student reading:** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning: With Applications in Python*. Springer. Selected classification and neural-network material. https://www.statlearning.com/

**Optional reading / instructor reference:** Molnar, C. *Interpretable Machine Learning*, 2nd ed. Coefficients and interpretation limitations. https://christophm.github.io/interpretable-ml-book/

**Dataset:** NIKHIL CHAUDHARY 7108. (n.d.). *Spam email dataset (Enron, TREC and others)*. [Kaggle](https://www.kaggle.com/datasets/nikhilchaudhary7108/spam-email-datasetenron-trec-and-others). Upstream rights and labels have not been independently audited. This notebook reads [the local prepared CSV](../../../00_Data/email_spam.csv); see [the repository README](../../../README.md) for preparation and limits.

**Official technical documentation:** [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html), [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html) and [safe model persistence](https://scikit-learn.org/stable/model_persistence.html).

**Environment:** [Repository requirements](../../../requirements.txt). Installed versions are displayed above; the footer records the runtime.

### Jupyter notebook - footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime

print(platform.platform())
print('Python:', platform.python_version())
print('Date:', datetime.now().isoformat(timespec='seconds'))